# 📊 E-Commerce AI Trend Analysis & Forecasting
**Project Track:**  AI for Market Trend Analysis (Module E)
**Author:** Aditya Kumar Bharti
**Date:** January 2026

---

## 1. Problem Definition & Objective

### 1.1 Problem Statement
In the dynamic landscape of Indian E-commerce, the biggest challenge for platforms is not just selling, but predicting **"What to sell, Where, and When?"**.

Currently, businesses struggle with:
1.  **Hyper-Local Demand Prediction:** It is extremely difficult to predict which product will sell most in a specific region (e.g., Heaters in North vs. Umbrellas in South) based on changing **Weather and Seasons**.
2.  **Inventory & Logistics Bottlenecks:** Without accurate regional forecasting, warehouses are either overstocked or understocked. This leads to delayed deliveries and higher return rates due to customer dissatisfaction.
3.  **Static Data Reliance:** Most systems work on old data. Modern businesses need a **"Live Dashboard"** that updates dynamically as every product is sold or returned, creating a smart loop of real-time intelligence.


### 1.2. Real-World Relevance
New businesses often lack the historical data to predict these trends. This project aims to build an **AI-driven Command Center** that combines Rule-Based Logic with Machine Learning to predict sales trends and segment customers for targeted marketing.

In the highly competitive Indian E-commerce market, inventory mismanagement during festive seasons (like Diwali, Big Billion Days) leads to massive revenue loss. Traditional forecasting methods fail to account for dynamic Indian seasonality and regional trends. Additionally, generic marketing wastes budget on low-value customers.

### 1.3 Objective
To build an AI-powered system that:
The objective of this project is to build an intelligent, daily-live AI Dashboard that empowers E-commerce platforms to-:

1.  **Provide Real-Time Intelligence:** Analyze sales and return data on a live basis to reflect the current market pulse.
2.  **Optimize Inventory by Region:** Predict demand spikes (e.g., Festive vs. Weather-driven) to stock up the right products in the right warehouses, ensuring faster delivery and extra facilities for customers.
3.  **Future Forecasting:** Use AI to predict upcoming trends, minimizing risks and maximizing operational efficiency.
4.  **Forecasts Sales:** uses a hybrid approach (Time-series logic + Domain Expert Rules) to predict sales spikes for 2026.
5.  **Segments Customers:** Uses Unsupervised Learning (K-Means Clustering) to identify High-Value (VIP) vs. Occasional buyers.
6.  **Provides Actionable Strategy:** Suggests inventory and marketing actions based on predictions.

In [21]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from datetime import datetime, timedelta
import calendar
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)

## 2. Data Understanding & Preparation

### 2.Dataset Source (Synthetic Engineering)
Since proprietary Indian e-commerce data is unavailable, I engineered a comprehensive **Synthetic Dataset** of **160,000+ rows**.
* **Logic:** The data generation script incorporates probability weights to simulate real-world events (e.g., 2.8x sales spike in Oct-Nov for Diwali).
* **Features:** Includes `Date`, `Customer_ID`, `Location`, `Category`, `Product`, `Quantity`, `Total_Sales`, etc.

### 2.2 Data Loading & Cleaning

In [22]:
def load_and_clean_data(file_path):
    try:
        df = pd.read_excel(file_path, engine='openpyxl')
        
        # Date Parsing
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df = df.dropna(subset=['Date'])
        
        # Schema Normalization
        if 'Quant' not in df.columns:
            if 'Quantity' in df.columns: df = df.rename(columns={'Quantity': 'Quant'})
            elif 'Qty' in df.columns: df = df.rename(columns={'Qty': 'Quant'})
            else: df['Quant'] = 1
            
        if 'City' not in df.columns and 'State' in df.columns:
            df['City'] = df['State']
            
        return df
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

# LOAD DATA
# Note: Ensure the path is correct relative to this notebook
file_path = "../data/India_Market_Trends_2025_Ultimate.xlsx"
df = load_and_clean_data(file_path)

print(f"Dataset Loaded Successfully: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Dataset Loaded Successfully: 160883 rows, 20 columns


,Date,Customer_ID,Age,Gender,State,City,Region,Location_Tier,Category,Sub_Category,Product_Name,Quant,Unit_Price,Total_Sales,Profit,Payment_Method,Status,Review_Text,Rating,Sentiment
0,2025-11-23,CUST_1488,55,Male,Delhi,Jaipur,North,Tier 2,Fashion,Women's Wear,Lehenga Choli,1,40741.0,40741.0,8148.2,COD,Delivered,Amazing product! Loved it,5,Positive
1,2025-02-17,CUST_1106,46,Male,Karnataka,Hyderabad,South,Tier 1,Mobiles & Accessories,Mobiles,Redmi Note 12,2,15436.0,30872.0,6174.4,Credit Card,Delivered,Amazing product! Loved it,5,Positive
2,2025-03-22,CUST_8527,56,Female,Odisha,Kolkata,East,Tier 1,Fashion,Footwear,Sports Shoes (Nike/Adidas),1,5940.0,5940.0,1188.0,UPI,Delivered,Amazing product! Loved it,6,Positive
3,2025-10-01,CUST_2307,67,Male,UP,Lucknow,North,Tier 2,Fashion,Footwear,Formal Shoes,2,1687.0,3374.0,674.8,COD,Delivered,Amazing product! Loved it,5,Positive
4,2025-04-29,CUST_2169,62,Female,UP,Jaipur,North,Tier 2,Books & Stationery,Stationery,Notebook Pack,1,543.0,543.0,-54.3,COD,Returned,"Product quality is very poor, returned immedia...",2,Negative


In [23]:
# Feature Engineering for Time Series Analysis
df['Month_Name'] = df['Date'].dt.month_name()
df['Month_Num'] = df['Date'].dt.month
df['Weekday'] = df['Date'].dt.day_name()
df['Year'] = df['Date'].dt.year

# Check for Missing Values
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
Date              0
Customer_ID       0
Age               0
Gender            0
State             0
City              0
Region            0
Location_Tier     0
Category          0
Sub_Category      0
Product_Name      0
Quant             0
Unit_Price        0
Total_Sales       0
Profit            0
Payment_Method    0
Status            0
Review_Text       0
Rating            0
Sentiment         0
Month_Name        0
Month_Num         0
Weekday           0
Year              0
dtype: int64


## 3. Model / System Design (The "Brain")

### a. Design Choice: Hybrid Architecture
We use a **Hybrid System**:
1.  **Forecasting:** A **Rule-Based Probabilistic Model**. Why? Because Indian festivals (Diwali, Holi) follow a lunar calendar and cause massive spikes (2.8x) that standard ARIMA models treat as noise. We hard-code these domain rules.
2.  **Segmentation:** **K-Means Clustering** (Unsupervised ML) to group customers based on spending behavior.

### b. The "Intelligent Strategy Engine"
This logic block takes a product category and suggests marketing tactics automatically.


In [24]:
def get_market_strategy(category, product_name, volume):
    category = str(category).lower()
    product_name = str(product_name).lower()
    strategy = "General Optimization"
    
    if any(x in product_name for x in ["earphone", "bud", "watch", "charger"]):
        strategy = "Tech Spec Optimization & Bundling (Highlight Playtime/Water Resistance)"
    elif "fashion" in category or "clothing" in category:
        strategy = "Visual Storytelling & Trend Keywords (Target 'Oversized', 'Streetwear')"
    elif "beauty" in category or "health" in category:
        strategy = "Ingredient Focus & Trust Badges ('Dermatologically Tested')"
    elif "home" in category:
        strategy = "Utility Demonstration (Before vs After Photos)"
        
    return strategy

# TEST THE ENGINE
sample_product = "Wireless Earbuds"
sample_cat = "Electronics"
print(f"AI Strategy for {sample_product}:")
print(get_market_strategy(sample_cat, sample_product, 100))

AI Strategy for Wireless Earbuds:
Tech Spec Optimization & Bundling (Highlight Playtime/Water Resistance)


## 4. Core Implementation: Customer Segmentation
We use **K-Means Clustering** to classify customers into 3 tiers:
1.  **VIP:** High Spend, High Volume.
2.  **Regular:** Average Spend.
3.  **Occasional:** Low Spend.


In [28]:
# Prepare Data: Group by Customer/State
group_col = 'Customer_ID' if 'Customer_ID' in df.columns else 'State'
cust_df = df.groupby(group_col).agg({'Total_Sales':'sum', 'Quant':'sum'}).reset_index()

# Normalize Data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(cust_df[['Total_Sales', 'Quant']])

# Apply K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cust_df['Cluster'] = kmeans.fit_predict(scaled_data)

# Map Clusters to Business Logic
cluster_avg = cust_df.groupby('Cluster')['Total_Sales'].mean().sort_values().index
cluster_map = {cluster_avg[0]: 'Occasional', cluster_avg[1]: 'Regular', cluster_avg[2]: 'VIP'}
cust_df['Segment'] = cust_df['Cluster'].map(cluster_map)

# Visualization
fig = px.scatter(cust_df, x="Quant", y="Total_Sales", color="Segment", 
                 title="Customer Segmentation: Volume vs Value",
                 color_discrete_map={'Occasional':'red', 'Regular':'orange', 'VIP':'green'})
fig.show()

print("Segment Distribution:")
print(cust_df['Segment'].value_counts())

Segment Distribution:
Segment
Occasional    77570
VIP            7662
Regular        5024
Name: count, dtype: int64


In [32]:
# 1. Visualize Customer Segmentation
if segmented_data is not None:
    fig = px.scatter(
        segmented_data, 
        x="Quant", 
        y="Total_Sales", 
        color="Segment",
        title="Customer Segmentation Analysis: Volume vs Value",
        color_discrete_map={
            'Occasional (Low Value)':'#EF553B', 
            'Regular (Mid Value)':'#FFA15A', 
            'High-Value VIP':'#00CC96'
        },
        template="plotly_dark"
    )
    fig.show()
else:
    print("Error: Segmentation data is not available.")

# 2. Visualize Sales Forecast
try:
    fig2 = go.Figure()
    
    # Historical & Forecast Lines
    fig2.add_trace(go.Scatter(
        x=forecast_df['Date'], 
        y=forecast_df['Predicted_Sales'], 
        mode='lines', 
        name='Forecast Prediction', 
        line=dict(color='#00F0FF')
    ))
    
    # Confidence Interval (Upper & Lower Bounds)
    fig2.add_trace(go.Scatter(
        x=forecast_df['Date'], 
        y=forecast_df['Upper_CI'], 
        mode='lines', 
        line=dict(width=0), 
        showlegend=False
    ))
    fig2.add_trace(go.Scatter(
        x=forecast_df['Date'], 
        y=forecast_df['Lower_CI'], 
        mode='lines', 
        line=dict(width=0), 
        fill='tonexty', 
        fillcolor='rgba(0, 240, 255, 0.2)', 
        name='Confidence Interval (90%)'
    ))
    
    fig2.update_layout(
        title="AI Sales Forecast (2026) - Seasonality Adjusted", 
        template="plotly_dark", 
        xaxis_title="Date", 
        yaxis_title="Revenue (INR)"
    )
    fig2.show()

except Exception as e:
    print(f"Visualization Error: {e}")

## 5. Core Implementation: Prediction Magic (Forecasting)
Here we implement the **Event-Aware Forecasting Logic**. This mimics the `predict_future` function in the dashboard.

In [29]:
# Calendar Logic
sale_calendar = {
    "January": [("Republic Day", 20, 26)],
    "March": [("Holi", 5, 12)],
    "October": [("Diwali", 20, 30)]
}

def simulate_prediction(day, month):
    base_rev = 1.8 # Lakhs
    multiplier = 1.0
    event_name = "Normal Day"
    
    # Check for Event
    if month in sale_calendar:
        for event, start, end in sale_calendar[month]:
            if start <= day <= end:
                event_name = event
                if "Diwali" in event: multiplier = 3.5
                elif "Republic" in event: multiplier = 2.5
                else: multiplier = 1.5
    
    predicted_rev = base_rev * multiplier
    return event_name, predicted_rev

# TEST CASE: Predicting Republic Day 2026
test_day = 26
test_month = "January"
event, rev = simulate_prediction(test_day, test_month)

print(f"Simulation for {test_day} {test_month}:")
print(f"Event Detected: {event}")
print(f"Predicted Revenue: ₹{rev:.2f} Lakhs")

Simulation for 26 January:
Event Detected: Republic Day
Predicted Revenue: ₹4.50 Lakhs


In [34]:
def generate_forecast_logic():
    # Simulation Parameters
    np.random.seed(42)
    base_sales = 150000
    today_dt = pd.Timestamp(datetime.now().date())
    
    # Create Future Dates (Next 1 Year)
    fore_dates = pd.date_range(today_dt, periods=365, freq='D')
    fore_sales, lower_ci, upper_ci = [], [], []

    # Logic: Indian Event Multipliers
    for d in fore_dates:
        multiplier = 1.0
        
        # Major Events Logic
        if d.month == 10: multiplier = 2.8  # Diwali Peak
        elif d.month == 8 and 10 <= d.day <= 15: multiplier = 2.0 # Independence Day
        elif d.month == 1 and 20 <= d.day <= 26: multiplier = 1.8 # Republic Day
        elif d.month == 11: multiplier = 1.5 # Wedding Season
        
        # Seasonality
        seasonal = 1.0
        if d.month in [4, 5, 6]: seasonal = 1.3 # Summer Demand
        
        # Add Random Noise for Realism
        noise = np.random.normal(0, 0.05)
        
        # Final Calculation
        pred_value = base_sales * multiplier * seasonal * (1 + noise)
        
        fore_sales.append(pred_value)
        lower_ci.append(pred_value * 0.9) # 10% Confidence Interval
        upper_ci.append(pred_value * 1.1)

    return pd.DataFrame({
        'Date': fore_dates,
        'Predicted_Sales': fore_sales,
        'Lower_CI': lower_ci,
        'Upper_CI': upper_ci
    })

# Execute Forecasting
forecast_df = generate_forecast_logic()
print("✅ Forecast Generated for next 365 days.")
display(forecast_df.head())

✅ Forecast Generated for next 365 days.


,Date,Predicted_Sales,Lower_CI,Upper_CI
0,2026-01-13,153725.356148,138352.820533,169097.891762
1,2026-01-14,148963.017741,134066.715967,163859.319515
2,2026-01-15,154857.664036,139371.897632,170343.430439
3,2026-01-16,161422.723923,145280.451531,177564.996315
4,2026-01-17,148243.849690,133419.464721,163068.234659


## 6. Evaluation & Analysis

### a. Clustering Quality (Silhouette Score)
We evaluate how well separated the customer segments are.

In [30]:
score = silhouette_score(scaled_data, cust_df['Cluster'])
print(f"Silhouette Score: {score:.3f}")
if score > 0.5:
    print("Verdict: Strong Cluster Separation (High Quality Segmentation)")
else:
    print("Verdict: Moderate Overlap (Expected in retail data)")

Silhouette Score: 0.681
Verdict: Strong Cluster Separation (High Quality Segmentation)


In [33]:
def perform_customer_segmentation(df):
    if df.empty: return None
    
    # 1. Feature Engineering: Group by Customer/State
    group_col = 'Customer ID' if 'Customer ID' in df.columns else 'State'
    cust_df = df.groupby(group_col).agg({
        'Total_Sales':'sum', 
        'Quant':'sum'
    }).reset_index()
    
    # 2. K-Means Implementation
    # We use 3 clusters: Occasional, Regular, High-Value
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    cust_df['Cluster'] = kmeans.fit_predict(cust_df[['Total_Sales', 'Quant']])
    
    # 3. Labeling Clusters based on Average Sales
    cluster_avg = cust_df.groupby('Cluster')['Total_Sales'].mean().sort_values().index
    cluster_map = {
        cluster_avg[0]: 'Occasional (Low Value)', 
        cluster_avg[1]: 'Regular (Mid Value)', 
        cluster_avg[2]: 'High-Value VIP'
    }
    cust_df['Segment'] = cust_df['Cluster'].map(cluster_map)
    
    return cust_df

# Run the model
segmented_data = perform_customer_segmentation(df)
print("✅ Segmentation Complete. Sample Results:")
display(segmented_data.head())

✅ Segmentation Complete. Sample Results:


,State,Total_Sales,Quant,Cluster,Segment
0,AP,1.939418e+07,3637,0,Occasional (Low Value)
1,Andhra Pradesh,2.130193e+06,745,0,Occasional (Low Value)
2,Assam,5.323186e+07,7309,0,Occasional (Low Value)
3,Bihar,4.786039e+07,7800,0,Occasional (Low Value)
4,Delhi,1.663261e+08,20969,2,High-Value VIP


### b. Visualizing the "Diwali Spike"
Validating that the historical data reflects Indian seasonality.

In [31]:
monthly_trend = df.groupby('Month_Num')['Total_Sales'].sum().reset_index()

fig = px.line(monthly_trend, x='Month_Num', y='Total_Sales', 
              title='Monthly Sales Trend (Look for Oct/Nov Spike)',
              markers=True)
fig.show()

## 7. Ethical Considerations & Conclusion

### Ethical AI
1.  **Synthetic Data:** The dataset is engineered for logic demonstration. Real-world application requires retraining on actual sales data.
2.  **Bias:** The "Occasional" segment label should not lead to exclusion from marketing, but rather different targeting (discounts vs early access).

### Conclusion
This project successfully demonstrates an **End-to-End AI Pipeline**. By combining **Synthetic Data Engineering** with **Hybrid Forecasting**, we created a system that doesn't just predict numbers, but understands the *context* of the Indian market.